# Module 4 - Class 2: Logistic Regression

Churn classification with logistic regression, metrics, and thresholds.


## 0. Setup


In [ ]:
# Import the libraries used in this notebook.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")


## 1. Load and Preprocess Data


In [ ]:
# Load and preprocess the churn dataset.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Standardize service columns
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Features and target
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Shape: {X.shape}")
print(f"Churn rate: {y.mean():.2%}")


In [ ]:
# Build preprocessing pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## 2. Train Logistic Regression


In [ ]:
# 2. Train Logistic Regression.
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")


## 3. Confusion Matrix


In [ ]:
# 3. Confusion Matrix.
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")


## 4. Classification Metrics


In [ ]:
# 4. Classification Metrics.
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")


In [ ]:
# Individual metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_proba)
}

for name, val in metrics.items():
    print(f"{name:>10}: {val:.4f}")


## 5. ROC Curve


In [ ]:
# 5. ROC Curve.
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_val = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 6. Coefficient Interpretation (Odds Ratios)

In logistic regression, exp(coefficient) gives the odds ratio: how much the odds of churn multiply when the feature increases by 1 unit.


In [ ]:
# Get feature names after encoding
ohe = pipe.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names

# Coefficients and odds ratios
coefs = pipe.named_steps['classifier'].coef_[0]
coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefs,
    'Odds Ratio': np.exp(coefs)
}).sort_values('Coefficient', ascending=False)

print("Top 10 features increasing churn:")
print(coef_df.head(10).to_string(index=False))
print()
print("Top 10 features decreasing churn:")
print(coef_df.tail(10).to_string(index=False))


In [ ]:
# Visualize top features
top_n = 15
top_features = pd.concat([coef_df.head(top_n // 2 + 1), coef_df.tail(top_n // 2)])

plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if c > 0 else '#1976d2' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.xlabel('Coefficient')
plt.title('Logistic Regression Coefficients (Red = increases churn)')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# Compare precision, recall, and F1 at several probability thresholds.
thresholds_to_test = [0.3, 0.4, 0.5]
threshold_results = []

for t in thresholds_to_test:
    y_pred_t = (y_proba >= t).astype(int)
    threshold_results.append({
        "Threshold": t,
        "Precision": precision_score(y_test, y_pred_t),
        "Recall": recall_score(y_test, y_pred_t),
        "F1": f1_score(y_test, y_pred_t),
    })

threshold_df = pd.DataFrame(threshold_results).round(4)
print(threshold_df.to_string(index=False))


## Threshold Conclusion

For a churn-prevention campaign I would choose the 0.3 threshold because it catches more likely churners and improves recall. This creates more false positives, but in retention work it is usually better to contact some extra customers than to miss many customers who are about to leave.
